In [ ]:
import pandas as pd
import json
import re
from tqdm.auto import tqdm
from openai import OpenAI

# Load API key
with open("key.txt", "r", encoding="utf-8") as f:
    api_key = f.read().strip()

client = OpenAI(api_key=api_key)

MODEL_NAME = "gpt-5.2-pro-2025-12-11"  # change if needed


In [ ]:
NAME = "Imru_Al_Qais_Poem_8"
CSV_PATH = NAME+".csv"

df = pd.read_csv(CSV_PATH)

# Combine Sadr + Ajuz into one verse
df["verse"] = df["Sadr"].fillna("") + " " + df["Ajuz"].fillna("")
df["verse"] = df["verse"].str.strip()

df.head()


In [ ]:
ELEMENT_FILES = {
    "Trope": "elements/Trope.txt",
    "Similie": "elements/Similie.txt",
    "Kenning": "elements/Kenning.txt",
    "Antonymy": "elements/Antonymy.txt"
}

ELEMENT_EXPLANATIONS = {}

for name, path in ELEMENT_FILES.items():
    with open(path, "r", encoding="utf-8") as f:
        ELEMENT_EXPLANATIONS[name] = f.read().strip()

ELEMENT_EXPLANATIONS.keys()


In [ ]:
def build_prompt(verse: str, element_name: str, explanation: str) -> str:
    return f"""
You are a STRICT JSON generator.

Task:
Determine whether the following Arabic verse contains the rhetorical element:
**{element_name}**

Definition:
{explanation}

Rules (VERY IMPORTANT):
- Output MUST be valid JSON
- Output ONLY JSON
- NO markdown
- NO comments
- Reasoning MUST be in Arabic
- If the element does NOT appear:
    - label = 0
    - span = "NONE"
    - reasoning = "" (EMPTY STRING)
- If the element DOES appear:
    - label = 1
    - span = exact words
    - reasoning = شرح مختصر بالعربية (قصير جدًا)

JSON schema (exact):
{{
  "label": 0 or 1,
  "span": "exact words from the verse" or "NONE",
  "reasoning": ""
}}

Verse:
{verse}
"""


In [ ]:
def safe_json_load(text):
    if not text or not isinstance(text, str):
        raise ValueError("Empty response")

    text = re.sub(r"^```json\s*", "", text.strip())
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    return json.loads(text)


In [6]:
for element_name, explanation in ELEMENT_EXPLANATIONS.items():
    df[f"{element_name}_label"] = None
    df[f"{element_name}_span"] = None
    df[f"{element_name}_reasoning"] = None

    print(f"\n🔍 Processing element: {element_name}")

    pbar = tqdm(
        total=len(df),
        desc=f"{element_name} | verses processed",
        leave=True
    )

    for idx in range(len(df)):
        verse = df.at[idx, "verse"]
        if not verse:
            pbar.update(1)
            continue

        prompt = build_prompt(verse, element_name, explanation)

        try:
            response = client.responses.create(
                model=MODEL_NAME,
                input=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
            )

            raw = response.output_text
            parsed = safe_json_load(raw)

            df.at[idx, f"{element_name}_label"] = parsed["label"]
            df.at[idx, f"{element_name}_span"] = parsed["span"]
            df.at[idx, f"{element_name}_reasoning"] = parsed["reasoning"]

        except Exception as e:
            df.at[idx, f"{element_name}_label"] = "ERROR"
            df.at[idx, f"{element_name}_span"] = "ERROR"
            df.at[idx, f"{element_name}_reasoning"] = str(e)

        pbar.update(1)

    pbar.close()

Trope | verses processed: 100%|██████████| 23/23 [16:30<00:00, 43.08s/it]



🔍 Processing element: Similie


Similie | verses processed: 100%|██████████| 23/23 [04:29<00:00, 11.73s/it]



🔍 Processing element: Kenning


Kenning | verses processed: 100%|██████████| 23/23 [13:05<00:00, 34.15s/it]



🔍 Processing element: Antonymy


Antonymy | verses processed: 100%|██████████| 23/23 [04:27<00:00, 11.61s/it]


In [7]:
df.head()


,Poet,Poem Title,Sadr,Ajuz,verse,Trope_label,Trope_span,Trope_reasoning,Similie_label,Similie_span,Similie_reasoning,Kenning_label,Kenning_span,Kenning_reasoning,Antonymy_label,Antonymy_span,Antonymy_reasoning
0,امرؤ القيس,قصيدة رقم 8,لِمَنِ الدّيَارُ غَشِيتُهَا بِسُحَامِ,فَعَمَايَتَينِ فَهَضْبِ ذِي أقْدَامِ,لِمَنِ الدّيَارُ غَشِيتُهَا بِسُحَامِ فَعَمَاي...,0,NONE,,0,NONE,,1,لِمَنِ الدّيَارُ,ذِكرُ «الديار» كنايةٌ عن أهلِها وسُكّانِها؛ لأ...,0,NONE,
1,امرؤ القيس,قصيدة رقم 8,فَصَفَا الأطِيطِ فَصَاحَتَينِ فَعَاشِمٍ,تَمْشِي النّعَاجُ بِهِ مَعَ الآرَامِ,فَصَفَا الأطِيطِ فَصَاحَتَينِ فَعَاشِمٍ تَمْشِ...,0,NONE,,0,NONE,,1,تَمْشِي النّعَاجُ بِهِ مَعَ الآرَامِ,كناية عن شدة الأمن والطمأنينة؛ إذ يُفهم من اجت...,0,NONE,
2,امرؤ القيس,قصيدة رقم 8,دَارٌ لِهِرٍّ وَالرَّبَابِ وَفَرْتَنى,وَلَمِيسَ قَبْلَ حَوَادِثِ الأيّامِ,دَارٌ لِهِرٍّ وَالرَّبَابِ وَفَرْتَنى وَلَمِيس...,1,حَوَادِثِ الأيّامِ,إسناد الحوادث إلى الأيام مجاز عقلي؛ فالأيام لا...,0,NONE,,1,حَوَادِثِ الأيّامِ,كناية عن نوائب الدهر ومصائبه؛ إذ يُشار إلى الش...,0,NONE,
3,امرؤ القيس,قصيدة رقم 8,عُوجَا عَلى الطَّلَلِ المُحِيلِ لَعَلَّنَا,نَبْكي الدّيارَ كما بكى ابنُ خِذامِ,عُوجَا عَلى الطَّلَلِ المُحِيلِ لَعَلَّنَا نَب...,1,نَبْكي الدّيارَ,مجاز؛ إذ البكاء يكون على أهل الديار وأحبتها لا...,1,نَبْكي الدّيارَ كما بكى ابنُ خِذامِ,شَبَّهَ بكاءَهم على الديار ببكاءِ ابنِ خِذام ب...,1,نَبْكي الدّيارَ,كناية عن البكاء على أهل الديار وأحبتها لا على ...,0,NONE,
4,امرؤ القيس,قصيدة رقم 8,دَارٌ لَهُم إذْ هُمْ لأَهْلِكَ جِيرَةٌ,إذْ تَسْتَبِيكَ بِواضِحٍ بَسَّامِ,دَارٌ لَهُم إذْ هُمْ لأَهْلِكَ جِيرَةٌ إذْ تَس...,1,تَسْتَبِيكَ بِواضِحٍ بَسَّامِ,استُعملت «تستبيك» بمعنى يأسرُك ويستميلُك لا بم...,0,NONE,,1,بِواضِحٍ بَسَّامِ,كناية عن جمال الثغر وبياض الأسنان؛ لم يصرّح با...,0,NONE,


In [8]:
OUTPUT_CSV = NAME+"analysis.csv"
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

OUTPUT_CSV


'Imru_Al_Qais_Poem_8analysis.csv'